In [1]:
import sys
sys.path.append('..')

import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, roc_auc_score

# --- Definir columnas por tipo ---

X_train = pd.read_parquet('../data/processed/X_train.parquet')
X_test  = pd.read_parquet('../data/processed/X_test.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')
y_test  = pd.read_parquet('../data/processed/y_test.parquet')

# --- Encodear target ---
y_train_enc = (y_train.iloc[:, 0] == 'new').astype(int)
y_test_enc  = (y_test.iloc[:, 0]  == 'new').astype(int)

# solo necesitás indicarle cuáles son categóricas
cat_features = ['category_id', 'city', 'warranty', 'listing_type_id', 
                'shipping_mode', 'buying_mode', 'state']

model = CatBoostClassifier(
    iterations=1000,
    random_seed=42,
    eval_metric='AUC',
    verbose=100
)

model.fit(
    X_train, y_train_enc,
    cat_features=cat_features,
    eval_set=(X_test, y_test_enc),
    early_stopping_rounds=50
)

y_pred       = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test_enc, y_pred, target_names=['used', 'new']))
print(f"AUC-ROC: {roc_auc_score(y_test_enc, y_pred_proba):.4f}")

Learning rate set to 0.096296
0:	test: 0.9087480	best: 0.9087480 (0)	total: 96.6ms	remaining: 1m 36s
100:	test: 0.9679115	best: 0.9679142 (99)	total: 3.64s	remaining: 32.4s
200:	test: 0.9703854	best: 0.9703854 (200)	total: 6.83s	remaining: 27.2s
300:	test: 0.9715124	best: 0.9715124 (300)	total: 10.1s	remaining: 23.3s
400:	test: 0.9722746	best: 0.9722746 (399)	total: 13.3s	remaining: 19.8s
500:	test: 0.9727815	best: 0.9727949 (494)	total: 16.5s	remaining: 16.4s
600:	test: 0.9730335	best: 0.9730350 (598)	total: 19.7s	remaining: 13.1s
700:	test: 0.9731886	best: 0.9732158 (681)	total: 23s	remaining: 9.79s
800:	test: 0.9733061	best: 0.9733233 (791)	total: 26.2s	remaining: 6.52s
900:	test: 0.9735166	best: 0.9735166 (900)	total: 30s	remaining: 3.29s
999:	test: 0.9735973	best: 0.9736090 (973)	total: 33.4s	remaining: 0us

bestTest = 0.9736089925
bestIteration = 973

Shrink model to first 974 iterations.
              precision    recall  f1-score   support

        used       0.90      0.92    

In [ ]:
import plotly.express as px

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=True)

fig = px.bar(
    feature_importance,
    x='importance',
    y='feature',
    orientation='h',
    title='Feature Importance — CatBoost',
    template='plotly_white'
)
fig.update_layout(height=700, width=800)
fig.show()

In [2]:
import optuna
from sklearn.metrics import precision_score
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train_enc, 
    test_size=0.15, 
    random_state=42, 
    stratify=y_train_enc
)

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength': trial.suggest_float('random_strength', 0, 1),
    }
    
    model = CatBoostClassifier(**params, random_seed=42, verbose=0)
    model.fit(
        X_tr, y_tr,
        cat_features=cat_features,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50
    )
    
    y_pred = model.predict(X_val)
    return precision_score(y_val, y_pred, pos_label=0)

sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Best params:", study.best_params)
print("Best precision (used):", study.best_value)

[I 2026-03-16 14:23:23,149] A new study created in memory with name: no-name-0c3f877c-c13c-401d-8dd1-ae7b464bd522


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-16 14:23:39,832] Trial 0 finished with value: 0.8960411471321695 and parameters: {'iterations': 1062, 'learning_rate': 0.2536999076681771, 'depth': 9, 'l2_leaf_reg': 6.387926357773329, 'bagging_temperature': 0.15601864044243652, 'random_strength': 0.15599452033620265}. Best is trial 0 with value: 0.8960411471321695.
[I 2026-03-16 14:24:03,796] Trial 1 finished with value: 0.8987873134328358 and parameters: {'iterations': 587, 'learning_rate': 0.19030368381735815, 'depth': 8, 'l2_leaf_reg': 7.372653200164409, 'bagging_temperature': 0.020584494295802447, 'random_strength': 0.9699098521619943}. Best is trial 1 with value: 0.8987873134328358.
[I 2026-03-16 14:24:50,368] Trial 2 finished with value: 0.8922647651526895 and parameters: {'iterations': 1749, 'learning_rate': 0.020589728197687916, 'depth': 5, 'l2_leaf_reg': 2.650640588680904, 'bagging_temperature': 0.3042422429595377, 'random_strength': 0.5247564316322378}. Best is trial 1 with value: 0.8987873134328358.
[I 2026-03-16

In [3]:
import plotly.express as px

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=True)

fig = px.bar(
    feature_importance,
    x='importance',
    y='feature',
    orientation='h',
    title='Feature Importance — CatBoost',
    template='plotly_white'
)
fig.update_layout(height=700, width=800)
fig.show()

In [42]:
# reentrenar el mejor modelo guardando las curvas
best_model = CatBoostClassifier(
    **study.best_params,
    random_seed=42,
    verbose=100,
    eval_metric='Precision',
    custom_metric=['Logloss', 'AUC'] 
)

# invertir encoding: used=1, new=0
y_tr_inv  = 1 - y_tr
y_val_inv = 1 - y_val

best_model.fit(
    X_tr, y_tr_inv,
    cat_features=cat_features,
    eval_set=(X_val, y_val_inv),
    early_stopping_rounds=40,
    plot=False
)
from plotly import graph_objects as go

# extraer curvas
evals_result = best_model.get_evals_result()

train_auc = evals_result['learn']['Logloss']
val_auc   = evals_result['validation']['Logloss']

fig = go.Figure()
fig.add_trace(go.Scatter(y=train_auc, name='Train Logloss', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=val_auc,   name='Val Logloss',   line=dict(color='red')))

fig.update_layout(
    title='Curvas de entrenamiento — CatBoost',
    xaxis_title='Iteración',
    yaxis_title='Logloss',
    template='plotly_white',
    height=500, width=900
)
fig.show()



0:	learn: 0.7962153	test: 0.7998192	best: 0.7998192 (0)	total: 161ms	remaining: 2m 49s
100:	learn: 0.8804593	test: 0.8866091	best: 0.8866440 (98)	total: 6.97s	remaining: 1m 6s
200:	learn: 0.8922620	test: 0.8911933	best: 0.8913817 (198)	total: 12.9s	remaining: 55.1s
Stopped by overfitting detector  (40 iterations wait)

bestTest = 0.8920173267
bestIteration = 224

Shrink model to first 225 iterations.


In [25]:
print(evals_result.keys())
print(evals_result['learn'].keys())
print(evals_result['validation'].keys())

dict_keys(['learn', 'validation'])
dict_keys(['Logloss'])
dict_keys(['Logloss', 'AUC'])


In [43]:
y_pred       = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test_enc, 1-y_pred, target_names=['used', 'new']))
print(f"AUC-ROC: {roc_auc_score(y_test_enc, y_pred_proba):.4f}")

              precision    recall  f1-score   support

        used       0.89      0.92      0.90      4594
         new       0.93      0.90      0.92      5406

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000

AUC-ROC: 0.0292


In [32]:
y_pred       = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test_enc, 1-y_pred, target_names=['used', 'new']))
print(f"AUC-ROC: {roc_auc_score(y_test_enc, y_pred_proba):.4f}")

              precision    recall  f1-score   support

        used       0.89      0.92      0.90      4594
         new       0.93      0.90      0.92      5406

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000

AUC-ROC: 0.0292


In [41]:
import numpy as np

print(classification_report(y_test_enc, 
                            np.where(y_pred_proba > 0.65, 0, 1),
                            target_names=['used', 'new']))


              precision    recall  f1-score   support

        used       0.92      0.87      0.90      4594
         new       0.90      0.94      0.92      5406

    accuracy                           0.91     10000
   macro avg       0.91      0.90      0.91     10000
weighted avg       0.91      0.91      0.91     10000



In [44]:
# guardar
best_model.save_model('../models/catboost_model.cbm')